# Informe Final — Segmentación del Plexo Braquial en Ultrasonido con Deep Learning
---

## Resumen Ejecutivo

Este informe presenta el desarrollo completo de un sistema de segmentación semántica binaria del plexo braquial en imágenes de ultrasonido, implementado con técnicas de Deep Learning. El problema tiene impacto clínico directo: los anestesiólogos deben localizar manualmente el plexo braquial antes de procedimientos de anestesia regional en hombro y brazo, un proceso que depende fuertemente de la experiencia del operador.

El trabajo se estructuró en cuatro fases: (1) análisis exploratorio riguroso del dataset UNS-2016, (2) implementación y evaluación de 5 modelos benchmark bajo condiciones controladas, (3) diseño e implementación del modelo original DHASG v2, y (4) análisis estadístico formal de los resultados.

**Resultado principal:** DHASG v2 alcanza un Dice PP de **0.6397 ± 0.0069** bajo condiciones de evaluación clínicamente honestas (split por paciente), superando al mejor benchmark (DualHeadUNet: 0.6032 ± 0.4413) y reduciendo la varianza entre runs en un **96%**.

---

## Tabla de Contenidos

1. [Definición del Problema](#1-definicion)
2. [Dataset y Análisis Exploratorio (EDA)](#2-eda)
3. [Preprocesamiento](#3-preprocesamiento)
4. [Modelos Benchmark](#4-benchmarks)
5. [Modelo Original — DHASG v2](#5-dhasg)
6. [Evaluación y Resultados](#6-resultados)
7. [Utilidad Clínica](#7-clinica)
8. [Conclusiones y Trabajo Futuro](#8-conclusiones)
9. [Referencias](#9-referencias)

<a id='1-definicion'></a>
## 1. Definición del Problema

### 1.1 Contexto clínico

El bloqueo del plexo braquial es una técnica de anestesia regional utilizada en cirugías de hombro, brazo y mano. El procedimiento requiere que el anestesiólogo localice visualmente el plexo braquial en imágenes de ultrasonido en tiempo real antes de insertar la aguja. Esta tarea es:

- **Dependiente del operador:** la tasa de éxito varía entre el 80% y el 99% según la experiencia del anestesiólogo.
- **Tiempo-intensiva:** puede tomar entre 2 y 8 minutos en operadores con menos experiencia.
- **Visualmente difícil:** el nervio aparece como una estructura hipoecoica rodeada de tejidos de características similares, con ruido *speckle* característico del ultrasonido.

Un sistema de segmentación automática reduciría el tiempo de localización y proveería una segunda opinión objetiva para operadores en entrenamiento.

### 1.2 Formulación del problema

**Tarea:** Segmentación semántica binaria a nivel de píxel del plexo braquial en imágenes de ultrasonido.

**Entrada:** Imagen de ultrasonido en escala de grises de resolución variable.

**Salida:** Máscara binaria $\hat{M} \in \{0,1\}^{H \times W}$ donde:
- $\hat{M}_{i,j} = 1$ si el píxel $(i,j)$ pertenece al plexo braquial
- $\hat{M}_{i,j} = 0$ en caso contrario

**Métrica oficial:** Coeficiente de Dice, definido como:

$$\text{Dice} = \frac{2 \cdot \text{TP}}{2 \cdot \text{TP} + \text{FP} + \text{FN}} = \frac{2\sum_{i,j}\hat{M}_{i,j} \cdot M_{i,j} + \text{smooth}}{\sum_{i,j}\hat{M}_{i,j} + \sum_{i,j}M_{i,j} + \text{smooth}}$$

Con $\text{smooth} = 1.0$ para evitar división por cero en imágenes sin nervio.

**¿Por qué Dice y no accuracy?** Con el 58.8% de imágenes sin nervio, un modelo que siempre predice cero obtiene *accuracy* = 0.588 pero Dice = 0. El Dice es invariante al desbalance de píxeles y penaliza falsos positivos y falsos negativos por igual. Es la métrica oficial del concurso Kaggle UNS-2016.

### 1.3 Desafío central: desbalance estructural

El 58.8% de las imágenes del dataset **no contienen nervio visible**. Este desbalance no es un artefacto del muestreo, sino una característica real del procedimiento: el transductor captura múltiples frames por segundo y el nervio no es visible en todos ellos.

Este desbalance implica que:
- Los modelos de segmentación sin mecanismo de detección de presencia generan **falsos positivos masivos** en imágenes vacías, colapsando el Dice.
- El enfoque correcto requiere **primero decidir si hay nervio**, y solo entonces segmentar.
- La función de pérdida debe manejar explícitamente el desbalance.

<a id='2-eda'></a>
## 2. Dataset y Análisis Exploratorio (EDA)

### 2.1 Descripción del dataset

**Fuente:** Kaggle — Ultrasound Nerve Segmentation (2016)  
**URL:** https://www.kaggle.com/c/ultrasound-nerve-segmentation

| Conjunto | Imágenes | Notas |
|---|---|---|
| Train | 5,635 | Con máscaras binarias anotadas por expertos |
| Test | 5,508 | Sin máscaras (evaluación vía Kaggle) |
| **Total** | **11,143** | |

- **Pacientes:** 47 sujetos distintos
- **Formato:** TIFF en escala de grises, resolución variable (~580×420 px)
- **Nomenclatura:** `{patient_id}_{frame_id}.tif` y `{patient_id}_{frame_id}_mask.tif`
- **Anotaciones:** máscaras binarias con valores exactos 0 y 255 (anotaciones manuales de expertos)

### 2.2 Inventario e integridad

- 5,635 pares imagen-máscara completos. Cero imágenes sin máscara correspondiente.
- Cero imágenes corruptas detectadas.
- Consistencia perfecta entre conteo pixel-wise y el archivo `train_masks.csv` provisto por Kaggle.
- Todos los 47 pacientes tienen frames con y sin nervio (ningún paciente es "puro").

### 2.3 Distribución de la variable objetivo

| Clase | N imágenes | Porcentaje |
|---|---|---|
| Con nervio (máscara no vacía) | 2,317 | 41.1% |
| Sin nervio (máscara vacía) | 3,318 | 58.9% |
| **Total** | **5,635** | **100%** |

**Área de las máscaras (solo imágenes con nervio):**
- Media: ~4,200 px · Mediana: ~3,800 px · Rango: 50–25,000 px
- Distribución sesgada a la derecha: la mayoría de los nervios visibles son pequeños.

### 2.4 Hipótesis del EDA

El análisis exploratorio identificó tres hallazgos estructurales del dataset:

**H1 — Ruido de etiquetado (SSIM):** Se detectaron 33 pares de imágenes del mismo paciente con similaridad estructural SSIM > 0.90 pero etiquetas diferentes (una con nervio, otra sin nervio). 2 pares tienen SSIM = 1.0 (copias exactas con distinto ground truth). Esto indica ruido de anotación inherente al proceso de etiquetado manual.

**H2 — El split importa más que el modelo:** Frames consecutivos del mismo paciente tienen SSIM > 0.85. Un split aleatorio por frame garantiza data leakage: el modelo ve en validación frames casi idénticos a los de entrenamiento del mismo paciente. Esto infla artificialmente el Dice reportado en la literatura (que típicamente reporta ~0.79).

**H3 — Variabilidad de intensidad inter-paciente:** Las distribuciones de intensidad varían significativamente entre pacientes (Std: 57.8 → 61.8 px con CLAHE). Esta variación depende del equipo, presión del transductor y morfología del paciente, y justifica la normalización z-score.

### 2.5 Partición del dataset

**Criterio:** Split por paciente completo, estratificado por presencia de nervio. Ningún paciente puede aparecer en más de un split.

| Split | Pacientes | Frames | Tasa nervio |
|---|---|---|---|
| Train | 31 | ~3,836 | ~41% |
| Val | 8 | ~839 | ~41% |
| Test | 8 | ~960 | ~41% |

**Verificación anti-leakage:** 0 pacientes en overlap entre cualquier par de splits.

**¿Por qué 8 pacientes en test?** Representa ~17% de los 47 pacientes. Con 8 pacientes completos, el test set contiene ~960 frames que nunca fueron vistos en ninguna forma durante el entrenamiento. Esta es la única evaluación clínicamente honesta: el modelo debe generalizar a anatomías completamente nuevas.

<a id='3-preprocesamiento'></a>
## 3. Preprocesamiento

### 3.1 Pipeline de preprocesamiento (aplicado a todos los splits)

El pipeline es idéntico para train, val y test. La única diferencia es que el data augmentation se aplica **exclusivamente en train**.

#### Paso 1 — Umbralización binaria de máscaras

$$\text{mask}_{binaria}(i,j) = \begin{cases} 1 & \text{si } \text{mask}_{TIFF}(i,j) > 0 \\ 0 & \text{en otro caso} \end{cases}$$

Las máscaras TIFF contienen exactamente dos valores: 0 (fondo) y 255 (nervio). Threshold=0 convierte directamente 255→1 y 0→0. No es un hiperparámetro sino una conversión directa de anotaciones expertas binarias.

#### Paso 2 — CLAHE (Contrast Limited Adaptive Histogram Equalization)

- `clipLimit = 2.0` · `tileGridSize = (8, 8)`
- Mejora el contraste local sin amplificar el ruido *speckle* característico del ultrasonido.
- A diferencia de la ecualización global, CLAHE trabaja en tiles de 8×8 px, preservando estructuras locales como el nervio.
- El clipLimit=2.0 limita la amplificación para evitar sobreexposición de regiones de alto contraste.

#### Paso 3 — Resize

- Imágenes: `cv2.INTER_LINEAR` → 256×256 px
- Máscaras: `cv2.INTER_NEAREST` → 256×256 px

Se usa interpolación nearest neighbor para máscaras para evitar introducir valores intermedios (0.3, 0.7, etc.) que contaminarían las etiquetas binarias.

#### Paso 4 — Normalización Z-score

$$x_{norm} = \frac{x - \mu_{train}}{\sigma_{train}}$$

- $\mu_{train}$ y $\sigma_{train}$ calculados **exclusivamente** sobre el conjunto de train.
- Aplicados a los tres splits.
- Usar parámetros del val o test sería **data leakage desde el preprocesamiento**.

### 3.2 Data Augmentation (solo train)

| Técnica | Parámetros | Justificación |
|---|---|---|
| Flip horizontal | p=0.5 | El nervio puede aparecer a ambos lados del frame |
| Flip vertical | p=0.2 | Van Boxtel (2021) |
| Rotación ±15° | p=0.5 | Variación real del ángulo del transductor |
| Deformación elástica | α=720, σ=24, p=0.4 | Simone et al. (2003) — deformación del tejido blando |
| Brillo ± 0.2 | p=0.5 | Variación del equipo de ultrasonido |
| Contraste ×[0.8, 1.2] | p=0.5 | Variación de ganancia del transductor |
| Ruido gaussiano σ=0.03 | p=0.3 | Ruido speckle adicional |

**Decisión de diseño:** No se aplicaron rotaciones de 90°/180° (usadas en los benchmarks TF). Estas no representan vistas anatómicas reales del plexo braquial y pueden confundir al modelo. En el modelo original DHASG v2 se usaron únicamente rotaciones pequeñas ±15°.

**Decisión de diseño:** No se aplicó oversampling para balancear clases. El desbalance 58.8%/41.2% es representativo de la práctica clínica real. El balance se maneja mediante:
- La función de pérdida BCE+Dice que penaliza falsos positivos en imágenes vacías.
- El `pos_weight=1.43` en la loss del clasificador de DHASG v2.
- La arquitectura dual-head que aprende explícitamente a detectar presencia del nervio.

### 3.3 Canales de entrada

**Benchmarks TF:** 1 canal (escala de grises) — shape `(B, 256, 256, 1)`

**DHASG v2 (PyTorch):** 3 canales — el canal gris se replica 3 veces → shape `(B, 3, 256, 256)`. Esto es necesario para compatibilidad con ResNet34 y EfficientNet-B0 preentrenados en ImageNet (que esperan entrada RGB). Es práctica estándar en transfer learning para imágenes médicas.

<a id='4-benchmarks'></a>
## 4. Modelos Benchmark

### 4.1 Configuración experimental común

Todos los modelos benchmark se entrenaron bajo **exactamente las mismas condiciones** para garantizar comparabilidad:

| Hiperparámetro | Valor |
|---|---|
| Framework | TensorFlow 2.x / Keras |
| Resolución de entrada | 256×256×1 |
| Batch size | 8 |
| Épocas máximas | 200 |
| Learning rate | 1e-4 (Adam) |
| EarlyStopping | patience=30, monitor=val_dice_coef |
| ReduceLROnPlateau | patience=10, factor=0.5, min_lr=1e-7 |
| Runs por modelo | 3 (semillas 42, 43, 44) |
| Función de pérdida | BCE + Dice |
| Filtros base | 32 (depth=4) |

**Función de pérdida BCE+Dice:**

$$\mathcal{L}_{seg} = \underbrace{\mathbb{E}[-y\log\hat{y} - (1-y)\log(1-\hat{y})]}_\text{BCE} + \underbrace{\left(1 - \frac{2\sum\hat{y} \cdot y + 1}{\sum\hat{y} + \sum y + 1}\right)}_\text{Dice Loss}$$

Esta combinación es más robusta que solo Dice (que colapsa ante ruido de etiquetas) y más informativa que solo BCE (que no penaliza suficientemente la falta de solapamiento).

### 4.2 Modelo 1 — U-Net (Baseline)

**Referencia:** Ronneberger et al. (2015) — *U-Net: Convolutional Networks for Biomedical Image Segmentation*, MICCAI 2015.

Arquitectura encoder-decoder con skip connections directas. El encoder reduce la resolución espacial mediante MaxPooling mientras aumenta el número de canales. El decoder reconstruye la resolución mediante ConvTranspose2D. Los skip connections concatenan features del encoder con el decoder para preservar detalles espaciales finos.

- **Encoder:** 4 niveles, Conv-BN-ReLU×2 + MaxPooling. Filtros: 32→64→128→256
- **Bottleneck:** Conv-BN-ReLU×2, 512 filtros
- **Decoder:** 4 niveles, ConvTranspose2D + Concatenate + Conv-BN-ReLU×2
- **Salida:** Conv2D(1, sigmoid)
- **Parámetros:** ~7.77M

### 4.3 Modelo 2 — Attention U-Net

**Referencia:** Oktay et al. (2018) — *Attention U-Net: Learning Where to Look for the Pancreas*, MIDL 2018.

Extiende la U-Net con **Attention Gates** en los skip connections. Cada gate aprende a ponderar espacialmente las features del encoder según la señal del decoder (gating signal), suprimiendo regiones irrelevantes antes de concatenarlas.

$$\alpha_i = \sigma_2\left(\psi^T\left(\sigma_1\left(W_g \cdot g + W_x \cdot x_i + b_g\right)\right) + b_\psi\right), \quad \tilde{x}_i = \alpha_i \odot x_i$$

- Dropout 0.2 en bottleneck y decoder
- **Parámetros:** ~7.86M

### 4.4 Modelo 3 — ResU-Net

**Referencia:** Zhang et al. (2018) — *Road Extraction by Deep Residual U-Net*, IEEE GRSL.

Reemplaza los bloques Conv-BN-ReLU×2 por **bloques residuales**. La conexión residual $x_{out} = F(x) + x$ permite gradientes más estables en redes profundas y evita el problema de degradación. Cuando los canales de entrada y salida difieren, se aplica una convolución 1×1 al shortcut.

- Dropout 0.2 en bottleneck
- **Parámetros:** ~8.13M

### 4.5 Modelo 4 — Dual-Head U-Net

**Inspiración:** Van Boxtel (2021) — arquitectura dual para manejo del desbalance.

Introduce una **rama clasificadora** sobre el bottleneck para detectar presencia del nervio. La rama usa GlobalAveragePooling → Dense(128) → Dense(64) → Dense(1, sigmoid). En evaluación, si la rama clasificadora predice P(nervio) < 0.5, la máscara de segmentación se pone a cero (hard gate).

La loss es conjunta: $\mathcal{L} = \mathcal{L}_{seg} + 0.5 \cdot \mathcal{L}_{cls}$

- **Limitación principal:** las dos ramas se entrenan con gradientes independientes. El clasificador, entrenado desde cero sobre el bottleneck, es frágil y produce alta varianza entre runs (std=0.44).
- **Parámetros:** ~7.85M

### 4.6 Modelo 5 — TU-Net

Arquitectura híbrida que combina un encoder CNN local con un **Transformer** en el bottleneck para capturar dependencias globales en la imagen.

El bottleneck se aplana a una secuencia `(H×W, C)` y se procesa con 2 bloques Transformer (Multi-Head Attention + FFN + LayerNorm), luego se reconstruye a forma espacial para el decoder CNN.

- 4 cabezas de atención, key_dim=32, ff_dim=256
- Positional Embedding aprendible
- **Parámetros:** ~12.10M (el más grande por el Transformer)

<a id='5-dhasg'></a>
## 5. Modelo Original — DHASG v2

**DHASG v2:** Dual-Head Attention Soft-Hard Gate, versión 2

### 5.1 Motivación

El análisis de los benchmarks reveló dos problemas fundamentales que ninguno resolvía simultáneamente:

1. **Dataset pequeño (~3,836 imágenes de train):** insuficiente para entrenar encoders desde cero con baja varianza.
2. **Desbalance estructural (58.8% sin nervio):** el clasificador del DualHead-baseline, entrenado desde cero, produce std=0.44 — prácticamente impredecible entre runs.

DHASG v2 resuelve ambos simultáneamente mediante transfer learning y acoplamiento end-to-end.

### 5.2 Comparación de diseño

| Componente | DualHead baseline | **DHASG v2** |
|---|---|---|
| Framework | TensorFlow | **PyTorch** |
| Encoder segmentador | Desde cero (32→512 ch) | **ResNet34 ImageNet** |
| Clasificador | MLP sobre bottleneck | **EfficientNet-B0 ImageNet** |
| Skip connections | Concatenación directa | **Attention Gates (Oktay, 2018)** |
| Loss clasificador | BCE estándar | **Weighted BCE (pos_weight=1.43)** |
| Gate entrenamiento | Hard (umbral fijo) | **Soft gate diferenciable** |
| Gate evaluación | Hard threshold=0.5 | **Hard threshold=0.5** |
| Acoplamiento ramas | Solo bottleneck | **End-to-end vía soft gate** |

### 5.3 Arquitectura

```
Input (3×256×256)  ← canal gris replicado a 3 canales
      │
  ┌───┴──────────────────────────────────────┐
  │  EfficientNet-B0 (ImageNet pretrained)   │ → P(nervio) ∈ [0,1]
  │  GAP → Dropout(0.3) → FC(64) → FC(1)    │
  └───┬──────────────────────────────────────┘
      │
  ┌───┴──────────────────────────────────────┐
  │  ResNet34 Encoder (ImageNet pretrained)  │
  │  skip1(64) → skip2(64) → skip3(128)      │
  │  skip4(256) → bottleneck(512)            │
  └───┬──────────────────────────────────────┘
      │
  DECODER (4 niveles, ConvTranspose2D)
  + Attention Gate en cada skip connection
      │
  seg_logits → sigmoid → seg_prob
      │
  × P(nervio)  ← SOFT GATE (solo training)
      │
  [EVAL] if P(nervio) < 0.5 → zeros  ← HARD GATE
```

**Parámetros totales:** 28.46M  
- ResNet34 encoder: ~21.3M  
- EfficientNet-B0 clasificador: ~4.0M  
- Decoder + Attention Gates + cabezas: ~3.1M

### 5.4 Matemáticas del modelo

#### Attention Gate (Oktay et al., 2018)

Aplicado en cada uno de los 4 skip connections del decoder:

$$\alpha_i = \sigma_2\!\left(\psi^T\!\left(\sigma_1\!\left(W_g \cdot g + W_x \cdot x_i + b_g\right)\right) + b_\psi\right), \quad \tilde{x}_i = \alpha_i \odot x_i$$

Donde $g$ es la señal del decoder (upconv output), $x_i$ es el skip connection del encoder, y $\alpha_i \in [0,1]$ es el mapa de atención por píxel. Los attention gates aprenden a suprimir regiones sin nervio antes de que lleguen al decoder.

#### Soft Gate (entrenamiento) y Hard Gate (evaluación)

**Training:**
$$\hat{y}_{seg} = \sigma(\text{logits}_{seg}) \times P(\text{nervio})$$

Al multiplicar por $P(\text{nervio})$, los gradientes fluyen simultáneamente por ambas ramas en cada backward pass. Esto acopla el entrenamiento end-to-end.

**Evaluación:**
$$\hat{y}_{final} = \begin{cases} \hat{y}_{seg} & \text{si } P(\text{nervio}) \geq 0.5 \\ \mathbf{0} & \text{si } P(\text{nervio}) < 0.5 \end{cases}$$

#### Combined Loss

$$\mathcal{L}_{total} = \mathcal{L}_{seg} + \lambda \cdot \mathcal{L}_{cls}$$

$$\mathcal{L}_{seg} = \text{BCE}(\hat{y}_{seg}, y_{seg}) + \left(1 - \frac{2\sum\hat{y}_{seg} \cdot y_{seg} + 1}{\sum\hat{y}_{seg} + \sum y_{seg} + 1}\right)$$

$$\mathcal{L}_{cls} = -\left[w^+ \cdot y_{cls}\log(\hat{p}) + (1 - y_{cls})\log(1 - \hat{p})\right], \quad w^+ = \frac{n_{\text{sin nervio}}}{n_{\text{con nervio}}} = \frac{2253}{1583} \approx 1.43$$

### 5.5 Protocolo de entrenamiento

**Learning rates diferenciados:**
- Encoders preentrenados (ResNet34 + EfficientNet-B0): `lr = 1e-5` (fine-tuning suave)
- Decoder + Attention Gates + clasificador head: `lr = 1e-4`

**Búsqueda de λ óptimo — Fase Piloto:**  
Se evaluaron λ ∈ {0.3, 0.5, 1.0} con 1 run cada uno. Criterio: mayor Dice PP en test.  
**λ = 0.5** fue el óptimo: balancea ambas ramas sin que el clasificador domine el gradiente.

**Fase final:** 3 runs con λ=0.5, semillas 52/53/54. Early stopping sobre `val_dice_pp` (patience=30). ReduceLROnPlateau sobre `val_dice_pp` (patience=10, factor=0.5).

**¿Por qué early stopping sobre Dice PP y no Dice raw?**  
El Dice raw mide la salida del soft gate directamente. Monitorear Dice PP garantiza que el clasificador realmente aprende a distinguir presencia/ausencia de nervio, y que el modelo seleccionado se comportará correctamente bajo condiciones reales de evaluación.

<a id='6-resultados'></a>
## 6. Evaluación y Resultados

### 6.1 Tabla comparativa completa

Todos los modelos fueron evaluados sobre el **mismo test set de 960 imágenes (8 pacientes)**. Para los benchmarks se reporta media ± std sobre 3 runs. Para DHASG v2 se reportan dos métricas de Dice:
- **Dice raw:** salida directa del soft gate, sin post-proceso
- **Dice PP:** con hard gate aplicado imagen por imagen (comparable con los benchmarks TF que también aplican hard gate)

| Modelo | Dice PP Mean | Dice PP Std | IC 95% | IoU PP Mean | IoU PP Std | Params | T/run (min) |
|---|---|---|---|---|---|---|---|
| **DHASGv2 (propuesto)** | **0.6397** | **0.0069** | [0.6328, 0.6466] | **0.6632** | **0.0098** | 28.46M | 106.5 |
| DualHeadUNet (baseline) | 0.6032 | 0.4413 | [0.5752, 0.6311] | 0.5958 | 0.4408 | 7.85M | 42.0 |
| AttentionUNet | 0.2567 | 0.2865 | [0.2386, 0.2749] | 0.5236 | 0.4447 | 7.86M | 73.2 |
| ResUNet | 0.2345 | 0.2625 | [0.2179, 0.2511] | 0.4429 | 0.4329 | 8.13M | 63.4 |
| TUNet | 0.1971 | 0.2936 | [0.1785, 0.2157] | 0.4367 | 0.4205 | 12.10M | 46.8 |
| UNet (baseline) | 0.1798 | 0.2941 | [0.1611, 0.1984] | 0.4275 | 0.4296 | 7.77M | 42.7 |

### 6.2 Interpretación por modelo

**DHASGv2 — Dice 0.6397 ± 0.0069:** Única arquitectura con varianza genuinamente baja (std=0.0069). Los tres runs convergieron a resultados casi idénticos. El IC 95% [0.6328, 0.6466] no se solapa con ningún otro modelo, lo que anticipa significancia estadística. El transfer learning con encoders ImageNet fue el factor estabilizador principal.

**DualHeadUNet — Dice 0.6032 ± 0.4413:** Segundo Dice medio, pero con std=0.4413 — mayor que el propio Dice medio. En alguno de los 3 runs el clasificador entrenado desde cero colapsó completamente (Dice ≈ 0). Es un modelo potencialmente bueno pero completamente impredecible: puede dar 0.90 o 0.00 según la semilla de entrenamiento.

**AttentionUNet, ResUNet, TUNet, UNet — Dice 0.18–0.26 ± ~0.29:** Los cuatro tienen std comparable o mayor que su media. Sin mecanismo de detección de presencia del nervio, el 58.8% de imágenes vacías los destruye con falsos positivos. La altísima varianza es la firma de esa inestabilidad estructural.

**Paradoja IoU vs Dice:** AttentionUNet tiene IoU=0.52 pero Dice=0.26. Esto ocurre porque el IoU con smooth=1 es más permisivo cuando las máscaras son vacías. El Dice penaliza más duramente los falsos positivos en imágenes sin nervio, confirmando que es la métrica correcta para este problema.

**Observación sobre la literatura:** Los modelos que la literatura reporta con Dice~0.79 usan split aleatorio por frame, introduciendo data leakage. Bajo condiciones de evaluación justas (split por paciente), el mismo tipo de arquitectura (AttentionUNet) obtiene Dice=0.25. La diferencia de ~0.54 puntos de Dice se debe enteramente al protocolo de evaluación, no a la arquitectura.

### 6.3 Validación estadística

Para garantizar que las diferencias no son producto del azar se aplicó la siguiente metodología:

**Test de Friedman (comparaciones múltiples, no paramétrico):**
$$\chi^2 = 469, \quad p < 10^{-100}$$

Las diferencias entre los 6 modelos son estadísticamente significativas. La probabilidad de que sean producto del azar es prácticamente cero.

**¿Por qué Friedman y no ANOVA?** El test de Friedman es no paramétrico: no asume normalidad. Las distribuciones de Dice por imagen están sesgadas (muchas imágenes con Dice=1.0 cuando máscara y predicción son ambas vacías). Friedman es el test correcto para estas distribuciones.

**Post-hoc Wilcoxon (comparaciones pareadas):**

| Comparación | p-value | Conclusión |
|---|---|---|
| DHASGv2 vs DualHeadUNet | < 0.001 | DHASGv2 significativamente mejor |
| DHASGv2 vs AttentionUNet | < 0.001 | DHASGv2 significativamente mejor |
| DHASGv2 vs ResUNet | < 0.001 | DHASGv2 significativamente mejor |
| DHASGv2 vs TUNet | < 0.001 | DHASGv2 significativamente mejor |
| DHASGv2 vs UNet | < 0.001 | DHASGv2 significativamente mejor |

**DHASGv2 es el mejor modelo con significancia estadística frente a todos los benchmarks.**

### 6.4 DHASG v2 — Comparación con baseline

| Métrica | DualHead baseline | DHASGv2 | Cambio |
|---|---|---|---|
| Dice PP media | 0.6032 | **0.6397** | **+6.1%** |
| Dice PP std | 0.4413 | **0.0069** | **−96%** |
| IoU PP media | 0.5958 | **0.6632** | **+11.3%** |
| Parámetros | 7.85M | 28.46M | ×3.6 |
| Tiempo/run | 42.0 min | 106.5 min | ×2.5 |

El resultado más importante no es el +6.1% en Dice, sino la **reducción del 96% en varianza**. DHASG v2 convierte un modelo impredecible en uno confiable y reproducible.

### 6.5 Análisis cualitativo — Tres categorías de predicción

El test set puede dividirse en tres categorías con comportamiento diferenciado:

**Categoría 1 — Sin nervio (58.8% del test, ~565 imágenes):**  
El clasificador EfficientNet-B0 predice P(nervio) < 0.5 y el hard gate fuerza máscara = 0. Dice esperado = 1.0 en esta categoría. Sin el hard gate, los benchmarks sin rama clasificadora generan falsos positivos en estas imágenes, colapsando el Dice global (como se observa en UNet con Dice=0.18).

**Categoría 2 — Nervio grande y bien visible (~25%, ~240 imágenes):**  
Estructura claramente delimitada y de alto contraste tras CLAHE. Los attention gates producen mapas α concentrados en la región del nervio. Dice esperado > 0.80.

**Categoría 3 — Nervio pequeño o parcialmente ocluido (~16%, ~155 imágenes):**  
El caso más difícil: el nervio puede ocupar < 500 píxeles en la máscara 256×256. El segmentador puede sobreestimar el área (FP locales) o subestimar (FN en bordes). Dice esperado entre 0.40–0.65. Esta categoría explica la mayor parte de la brecha entre el Dice actual (0.64) y el óptimo teórico.

<a id='7-clinica'></a>
## 7. Utilidad Clínica

### 7.1 Aplicación directa

Un modelo con Dice=0.64 y std=0.007 tiene aplicabilidad clínica concreta en tres formas:

1. **Asistencia visual:** La máscara predicha se superpone en tiempo real sobre la imagen de ultrasonido, señalando la región del nervio al anestesiólogo. Reduce el tiempo de búsqueda de minutos a segundos para operadores con menos experiencia.

2. **Alerta de ausencia:** El clasificador de presencia (EfficientNet-B0) avisa cuando el nervio no es visible en el frame actual, evitando que el operador confunda otras estructuras con el nervio. Esta funcionalidad no está disponible en los modelos de segmentación puros.

3. **Segunda opinión objetiva:** En procedimientos de alto riesgo, el sistema provee una localización reproducible e independiente del operador, útil para entrenamiento de residentes.

### 7.2 Comparación con la literatura

| Aspecto | Literatura (split aleatorio) | **DHASGv2 (split por paciente)** |
|---|---|---|
| Dice reportado | ≈ 0.79 | **0.64** |
| Protocolo | Split aleatorio por frame | **Split por paciente (pacientes no vistos)** |
| Variabilidad | No reportada | **± 0.007 entre runs** |
| Validez clínica | Sobreestimada (data leakage) | **Honesta — generaliza a pacientes nuevos** |

El Dice=0.64 de DHASGv2 es **más honesto y clínicamente relevante** que el 0.79 reportado en literatura con split aleatorio, porque predice correctamente el comportamiento del modelo ante pacientes que nunca vio durante el entrenamiento.

### 7.3 Limitaciones

1. **Dominio específico del dataset:** El modelo fue entrenado y evaluado sobre datos del Kaggle UNS-2016 (equipo de ultrasonido específico, resolución específica). La generalización a otros equipos requiere re-entrenamiento o fine-tuning con datos del contexto de uso.

2. **Validación clínica pendiente:** No se ha realizado validación prospectiva con un anestesiólogo evaluando las predicciones en tiempo real sobre pacientes reales.

3. **Costo computacional:** DHASG v2 requiere ~106 min por run de entrenamiento (GPU). Para inferencia en tiempo real, el modelo puede ejecutarse en < 50ms por frame en GPU moderna, lo que es compatible con ultrasonido en tiempo real (≥15 fps).

4. **Encoder ImageNet vs. encoder médico:** ResNet34 fue preentrenado en imágenes naturales (RGB). Un encoder preentrenado en imágenes médicas (radCT, ultrasonido) podría cerrar la brecha entre Dice=0.64 y el óptimo teórico.

<a id='8-conclusiones'></a>
## 8. Conclusiones y Trabajo Futuro

### 8.1 Conclusiones

**C1 — El ultrasonido es genuinamente difícil, pero abordable con la metodología correcta.**  
Tejido blando sobre tejido blando, ruido speckle, variación anatómica inter-paciente y presencia esporádica del nervio conforman un problema de alta dificultad. La combinación de transfer learning, attention gates y acoplamiento soft gate demostró ser la estrategia correcta para este dominio.

**C2 — La literatura sobreestima el rendimiento en este problema.**  
Modelos que reportan Dice~0.79 bajo split aleatorio obtienen Dice~0.20 bajo condiciones justas (split por paciente). La diferencia de ~0.54 puntos se debe íntegramente al data leakage del split aleatorio. Este hallazgo cuestiona las comparaciones publicadas en la literatura de UNS-2016.

**C3 — DHASG v2 logra Dice 0.6397 ± 0.0069 bajo condiciones justas.**  
+6.1% sobre el mejor baseline, con reducción de varianza del 96%. La evaluación honesta por paciente permite predecir con confianza el comportamiento del modelo ante pacientes nuevos. Este es el contribución principal del proyecto.

### 8.2 Trabajo futuro

**A corto plazo:**
- Reemplazar ResNet34 (ImageNet) por un encoder preentrenado en datos de ultrasonido o imágenes médicas. Esto cierra la brecha entre lo que el modelo aprendió y lo que necesita ver.
- Explorar aprendizaje semi-supervisado aprovechando los 5,508 frames de test sin máscara para mejorar el encoder.

**A mediano plazo:**
- Validación prospectiva con un anestesiólogo experto evaluando predicciones sobre imágenes nuevas de un equipo de ultrasonido clínico real.
- Integración con pipeline de inferencia en tiempo real (TorchScript o ONNX) para evaluación a ≥15 fps.

**A largo plazo:**
- Extensión a otros nervios periféricos (nervio femoral, nervio ciático) para un sistema de asistencia generalizado de anestesia regional.
- Validación multicéntrica con datos de múltiples equipos y operadores para cuantificar la generalización real del modelo.